# Pengambilan Data, Preprocessing, Sentimen, dan Topic Modeling

Final Project Analisis Media Sosial: *Pemetaan Polarisasi Komunitas dan Dinamika Diskusi Terkait Tren Autonomous Coding Agent pada Kanal YouTube Teknologi Melalui Social Network Analysis*.

Notebook ini menghasilkan dataset CSV yang dipakai untuk analisis sentimen dan topik, sekaligus untuk analisis jaringan, deteksi komunitas, dan visualisasi Gephi.

**Alur:** konfigurasi → ambil data dari YouTube Data API v3 → skema dan preprocessing → analisis sentimen → topic modeling → ekspor CSV → pengecekan.

**Cara menjalankan:**
1. Buka notebook ini dari folder utama proyek (`anmedsos/`).
2. Salin `.env.example` menjadi `.env`, lalu isi `YOUTUBE_API_KEY`.
3. Jalankan semua sel dari atas (*Run All*).

Data mentah dan cache disimpan per video, jadi notebook bisa dijalankan ulang tanpa memanggil API lagi. Kalau kuota API habis di tengah jalan, jalankan lagi besok dan video yang sudah selesai tidak diambil ulang.

## 0. Instalasi library (sekali saja)

In [ ]:
%pip install -r requirements.txt

## 1. Setup dan konfigurasi

In [1]:
import os
import re
import json
import time
import random
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
RAW_COMMENTS_DIR = DATA_RAW / "comments"
DATA_PROC = ROOT / "data" / "processed"
CACHE = ROOT / "data" / "cache"
for folder in (RAW_COMMENTS_DIR, DATA_PROC, CACHE):
    folder.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")
pd.set_option("display.max_colwidth", 120)
print("Folder proyek:", ROOT)

Folder proyek: /ansosmed


/ansosmed/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Crawling
VIDEO_IDS = [
    "AgyJv2Qelwk",
    "PLKrSVuT-Dg",
    "tNmgmwEtoWE",
    "-5LfRL82Jck",
    "JMYspR42HFM",
    "b0fv54xGOwY",
]

FORCE_RECRAWL = False        # True = ambil ulang semua komentar dari API (data lama ditimpa)
FORCE_RECOMPUTE_NLP = False  # True = hitung ulang sentimen dan embedding

MIN_WORDS_NLP = 3            # komentar < 3 kata tidak dipakai untuk sentimen/topik
SENTIMENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

assert all(len(v) == 11 for v in VIDEO_IDS), "Cek lagi VIDEO_IDS: tiap ID harus 11 karakter, tanpa 'watch?v='"

## 2. Pengambilan data dari YouTube Data API v3

- `videos.list` untuk metadata video (judul, kanal, jumlah view/like/komentar).
- `commentThreads.list` untuk komentar utama. Endpoint ini hanya menyertakan maksimal 5 balasan per komentar.
- `comments.list` dengan `parentId` untuk mengambil semua balasan ketika balasannya lebih dari 5.

Hasil mentah disimpan per video di `data/raw/comments/<video_id>.csv`.

In [3]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

RAW_COLUMNS = [
    "video_id", "comment_id", "parent_id", "author_channel_id", "author_display_name",
    "text_display", "text_original", "like_count", "published_at", "updated_at", "total_reply_count",
]


def get_client():
    key = os.getenv("YOUTUBE_API_KEY")
    if not key:
        raise RuntimeError("YOUTUBE_API_KEY belum diisi. Salin .env.example menjadi .env lalu isi API key.")
    return build("youtube", "v3", developerKey=key, cache_discovery=False)


def call_api(request, max_retry=5):
    for attempt in range(max_retry):
        try:
            return request.execute()
        except HttpError as e:
            msg = str(e)
            if "quotaExceeded" in msg:
                raise RuntimeError("Kuota harian API habis. Jalankan lagi besok; video yang sudah selesai tidak diambil ulang.") from e
            if e.resp.status in (429, 500, 503) or "rateLimitExceeded" in msg:
                time.sleep(2 ** attempt)
                continue
            raise
    raise RuntimeError("Request API gagal setelah beberapa kali percobaan.")


def parse_comment(item, video_id, parent_id=None, total_reply_count=0):
    s = item["snippet"]
    return {
        "video_id": video_id,
        "comment_id": item["id"],
        "parent_id": parent_id,
        "author_channel_id": (s.get("authorChannelId") or {}).get("value"),
        "author_display_name": s.get("authorDisplayName"),
        "text_display": s.get("textDisplay"),
        "text_original": s.get("textOriginal"),
        "like_count": s.get("likeCount", 0),
        "published_at": s.get("publishedAt"),
        "updated_at": s.get("updatedAt"),
        "total_reply_count": total_reply_count,
    }


def fetch_videos(yt, video_ids):
    rows = []
    for i in range(0, len(video_ids), 50):
        res = call_api(yt.videos().list(part="snippet,statistics", id=",".join(video_ids[i:i + 50])))
        for v in res.get("items", []):
            s, st = v["snippet"], v.get("statistics", {})
            rows.append({
                "video_id": v["id"],
                "channel_id": s.get("channelId"),
                "channel_title": s.get("channelTitle"),
                "title": s.get("title"),
                "video_published_at": s.get("publishedAt"),
                "view_count": int(st.get("viewCount", 0)),
                "like_count": int(st.get("likeCount", 0)),
                "comment_count": int(st.get("commentCount", 0)),
            })
    found = {r["video_id"] for r in rows}
    missing = [v for v in video_ids if v not in found]
    if missing:
        print("Peringatan: video tidak ditemukan atau privat:", missing)
    return pd.DataFrame(rows)


def fetch_replies(yt, parent_id, video_id):
    rows = []
    request = yt.comments().list(part="snippet", parentId=parent_id, maxResults=100, textFormat="plainText")
    while request is not None:
        res = call_api(request)
        rows += [parse_comment(c, video_id, parent_id=parent_id) for c in res.get("items", [])]
        request = yt.comments().list_next(request, res)
    return rows


def fetch_video_comments(yt, video_id):
    rows = []
    request = yt.commentThreads().list(
        part="snippet,replies", videoId=video_id, maxResults=100, textFormat="plainText", order="time"
    )
    while request is not None:
        res = call_api(request)
        for thread in res.get("items", []):
            top = thread["snippet"]["topLevelComment"]
            n_reply = thread["snippet"].get("totalReplyCount", 0)
            rows.append(parse_comment(top, video_id, total_reply_count=n_reply))
            inline = thread.get("replies", {}).get("comments", [])
            if n_reply > len(inline):
                rows += fetch_replies(yt, top["id"], video_id)
            else:
                rows += [parse_comment(c, video_id, parent_id=top["id"]) for c in inline]
        request = yt.commentThreads().list_next(request, res)
    return rows

In [4]:
videos_path = DATA_RAW / "videos_raw.csv"
yt = None

if FORCE_RECRAWL or not videos_path.exists():
    yt = get_client()
    videos_raw = fetch_videos(yt, VIDEO_IDS)
    videos_raw["crawled_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    videos_raw.to_csv(videos_path, index=False)

videos_raw = pd.read_csv(videos_path, dtype={"video_id": str, "channel_id": str})
todo = [v for v in videos_raw["video_id"] if FORCE_RECRAWL or not (RAW_COMMENTS_DIR / f"{v}.csv").exists()]

for vid in tqdm(todo, desc="Mengambil komentar"):
    yt = yt or get_client()
    try:
        rows = fetch_video_comments(yt, vid)
    except HttpError as e:
        if "commentsDisabled" in str(e):
            print(f"Komentar dinonaktifkan pada video {vid}, dilewati.")
            rows = []
        else:
            raise
    df_vid = pd.DataFrame(rows, columns=RAW_COLUMNS)
    df_vid["crawled_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    df_vid.to_csv(RAW_COMMENTS_DIR / f"{vid}.csv", index=False)

if not todo:
    print("Memakai data mentah yang sudah ada. Set FORCE_RECRAWL = True untuk mengambil ulang.")

raw = pd.concat(
    [pd.read_csv(RAW_COMMENTS_DIR / f"{v}.csv", dtype=str) for v in videos_raw["video_id"]],
    ignore_index=True,
)
summary = raw.groupby("video_id").agg(rows=("comment_id", "size"), replies=("parent_id", "count"))
display(videos_raw[["video_id", "channel_title", "title", "comment_count"]].merge(summary, on="video_id"))
print("Total baris mentah:", len(raw))

Mengambil komentar: 100%|██████████| 6/6 [01:14<00:00, 12.34s/it]


,video_id,channel_title,title,comment_count,rows,replies
0,AgyJv2Qelwk,Fireship,AI just officially took our jobs… I hate you Devin,9248,9247,3721
1,PLKrSVuT-Dg,Fireship,How to make vibe coding not suck…,1208,1221,346
2,tNmgmwEtoWE,Internet of Bugs,"Debunking Devin: ""First AI Software Engineer"" Upwork lie exposed!",1177,1188,457
3,-5LfRL82Jck,Theo - t3․gg,I’m addicted to Claude Code (i get it now),689,689,238
4,JMYspR42HFM,Theo - t3․gg,Claude Code vs Codex vs Cursor (an honest comparison),518,525,133
5,b0fv54xGOwY,Theo - t3․gg,"My Thoughts On ""Vibe Coding"" (And Prime)",471,471,124


Total baris mentah: 13341


## 3. Skema data dan preprocessing

Langkah:
1. Ubah tipe data, buang duplikat, buang komentar kosong atau tanpa akun.
2. Tentukan **target balasan** (siapa membalas siapa). Di YouTube semua balasan menempel ke komentar utama, padahal sering kali yang dibalas adalah pembalas lain yang disebut di awal teks (`@handle`). Jika `@handle` di awal balasan cocok dengan penulis lain di thread yang sama, target = penulis tersebut (`reply_target_method = mention`). Jika tidak, target = penulis komentar utama (`parent`).
3. Tandai balasan ke diri sendiri (`is_self_reply`) dan komentar dari pemilik kanal (`is_channel_owner`).
4. **Anonimisasi**: ID kanal diganti `user_00001`, dst. Nama tampilan dibuang, dan `@handle` di teks diganti `@user`.
5. Bersihkan teks dan deteksi bahasa. Sentimen dan topik hanya untuk komentar berbahasa Inggris dengan minimal `MIN_WORDS_NLP` kata. Jaringan tetap memakai semua komentar.

In [5]:
df = raw.copy()
n_awal = len(df)

df["like_count"] = pd.to_numeric(df["like_count"], errors="coerce").fillna(0).astype(int)
df["total_reply_count"] = pd.to_numeric(df["total_reply_count"], errors="coerce").fillna(0).astype(int)
df["published_at"] = pd.to_datetime(df["published_at"], utc=True)
df["updated_at"] = pd.to_datetime(df["updated_at"], utc=True)

df = df.drop_duplicates("comment_id")
df = df[df["text_display"].fillna("").str.strip().ne("")]
df = df[df["author_channel_id"].notna()]
df["is_reply"] = df["parent_id"].notna()
df = df.sort_values(["video_id", "published_at", "comment_id"]).reset_index(drop=True)

print(f"Baris awal: {n_awal} | setelah dibersihkan: {len(df)} | dibuang: {n_awal - len(df)}")

Baris awal: 13341 | setelah dibersihkan: 13307 | dibuang: 34


In [6]:
HANDLE_RE = re.compile(r"^\s*@([^\s]+)")


def norm_handle(h):
    return str(h).strip().lstrip("@").rstrip(".,:;!?)").lower()


top_author = df.loc[~df["is_reply"]].set_index("comment_id")["author_channel_id"]
top_name = df.loc[~df["is_reply"]].set_index("comment_id")["author_display_name"]

reply_target, target_method = {}, {}
replies = df[df["is_reply"]].sort_values(["published_at", "comment_id"])
for parent_id, thread in replies.groupby("parent_id", sort=False):
    known = {}
    if parent_id in top_author.index:
        known[norm_handle(top_name[parent_id])] = top_author[parent_id]
    for cid, author, name, text in thread[["comment_id", "author_channel_id", "author_display_name", "text_display"]].itertuples(index=False):
        target, method = top_author.get(parent_id), "parent"
        m = HANDLE_RE.match(str(text))
        if m:
            h = norm_handle(m.group(1))
            if h in known and known[h] != author:
                target, method = known[h], "mention"
        reply_target[cid], target_method[cid] = target, method
        if pd.notna(name):
            known[norm_handle(name)] = author

df["reply_to_author_id"] = df["comment_id"].map(reply_target)
df["reply_target_method"] = df["comment_id"].map(target_method)
df["is_self_reply"] = df["is_reply"] & (df["author_channel_id"] == df["reply_to_author_id"])
owner = videos_raw.set_index("video_id")["channel_id"]
df["is_channel_owner"] = df["author_channel_id"] == df["video_id"].map(owner)

print(df.loc[df["is_reply"], "reply_target_method"].value_counts(dropna=False))
print("Balasan ke diri sendiri:", int(df["is_self_reply"].sum()))
print("Balasan tanpa komentar utama (target kosong):", int((df["is_reply"] & df["reply_to_author_id"].isna()).sum()))

reply_target_method
parent     3937
mention    1054
Name: count, dtype: int64
Balasan ke diri sendiri: 122
Balasan tanpa komentar utama (target kosong): 0


In [7]:
# Anonimisasi: urutan ID diurutkan alfabetis supaya pemetaan selalu sama untuk data yang sama
all_authors = sorted(set(df["author_channel_id"]) | set(df["reply_to_author_id"].dropna()))
id_map = {a: f"user_{i:05d}" for i, a in enumerate(all_authors, start=1)}
pd.Series(id_map, name="user_id").rename_axis("author_channel_id").to_csv(DATA_RAW / "user_id_map.csv")  # rahasia, tidak di-upload

df["user_id"] = df["author_channel_id"].map(id_map)
df["reply_to_user_id"] = df["reply_to_author_id"].map(id_map)

URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@[^\s]+")


def mask_mentions(t):
    return MENTION_RE.sub("@user", str(t))


def text_for_sentiment(t):
    t = URL_RE.sub("http", str(t))
    return " ".join(MENTION_RE.sub("@user", t).split())


def text_for_topic(t):
    t = URL_RE.sub(" ", str(t))
    t = MENTION_RE.sub(" ", t)
    t = re.sub(r"[^\w\s'.,!?-]", " ", t)
    return " ".join(t.split())


df["text"] = df["text_display"].map(mask_mentions)
df["text_sentiment"] = df["text_display"].map(text_for_sentiment)
df["text_clean"] = df["text_display"].map(text_for_topic)
df["n_words"] = df["text_clean"].str.split().str.len().fillna(0).astype(int)

In [8]:
from langdetect import DetectorFactory, detect, LangDetectException

DetectorFactory.seed = 0  # hasil deteksi bahasa jadi konsisten


def detect_lang(t):
    if len(t.split()) < MIN_WORDS_NLP:
        return "und"
    try:
        return detect(t)
    except LangDetectException:
        return "und"


df["lang"] = [detect_lang(t) for t in tqdm(df["text_clean"], desc="Deteksi bahasa")]
df["use_for_nlp"] = (df["lang"] == "en") & (df["n_words"] >= MIN_WORDS_NLP)

print(df["lang"].value_counts().head(8))
print(f"Dipakai untuk sentimen/topik: {int(df['use_for_nlp'].sum())} dari {len(df)} komentar")

Deteksi bahasa: 100%|██████████| 13307/13307 [00:10<00:00, 1234.40it/s]

lang
en     11659
und      936
nl        74
af        73
fr        64
no        60
da        41
it        36
Name: count, dtype: int64
Dipakai untuk sentimen/topik: 11659 dari 13307 komentar


## 4. Analisis sentimen

Model: `cardiffnlp/twitter-roberta-base-sentiment-latest` (RoBERTa yang dilatih pada teks media sosial), dengan 3 label: negative, neutral, positive.

- `sentiment_label` = label dengan probabilitas tertinggi.
- `sentiment_score` = p_positive − p_negative (rentang −1 sampai 1).

Hasil disimpan di `data/cache/sentiment.csv`, jadi model tidak perlu dijalankan ulang.

In [9]:
import torch
from transformers import pipeline

torch.manual_seed(SEED)
sent_cache = CACHE / "sentiment.csv"
PROB_COLS = ["p_negative", "p_neutral", "p_positive"]

nlp_rows = df.loc[df["use_for_nlp"], ["comment_id", "text_sentiment"]]
if sent_cache.exists() and not FORCE_RECOMPUTE_NLP:
    sent = pd.read_csv(sent_cache, dtype={"comment_id": str})
else:
    sent = pd.DataFrame(columns=["comment_id"] + PROB_COLS)

todo = nlp_rows[~nlp_rows["comment_id"].isin(sent["comment_id"])]
if len(todo):
    clf = pipeline(
        "text-classification", model=SENTIMENT_MODEL, tokenizer=SENTIMENT_MODEL,
        top_k=None, device=0 if torch.cuda.is_available() else -1,
    )
    new_rows, chunk = [], 256
    for i in tqdm(range(0, len(todo), chunk), desc="Sentimen"):
        part = todo.iloc[i:i + chunk]
        outputs = clf(part["text_sentiment"].tolist(), batch_size=32, truncation=True, max_length=512)
        for cid, scores in zip(part["comment_id"], outputs):
            p = {s["label"].lower(): float(s["score"]) for s in scores}
            new_rows.append({"comment_id": cid, "p_negative": p["negative"], "p_neutral": p["neutral"], "p_positive": p["positive"]})
    sent = pd.concat([x for x in (sent, pd.DataFrame(new_rows)) if len(x)], ignore_index=True)
    sent.to_csv(sent_cache, index=False)

sent = sent[sent["comment_id"].isin(nlp_rows["comment_id"])].copy()
sent[PROB_COLS] = sent[PROB_COLS].astype(float)
sent["sentiment_label"] = sent[PROB_COLS].idxmax(axis=1).str.replace("p_", "", regex=False)
sent["sentiment_score"] = sent["p_positive"] - sent["p_negative"]

df = df.drop(columns=PROB_COLS + ["sentiment_label", "sentiment_score"], errors="ignore").merge(sent, on="comment_id", how="left")
print(df["sentiment_label"].value_counts(dropna=False))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 55051.27it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Sentimen: 100%|██████████| 46/46 [00:10<00:00,  4.23it/s]


sentiment_label
negative    4974
neutral     4425
positive    2260
NaN         1648
Name: count, dtype: int64


## 5. Topic modeling (BERTopic)

Parameter dikunci di sel di bawah (`random_state` UMAP = SEED) supaya hasil topik sama setiap dijalankan. Topik `-1` adalah *outlier* (komentar yang tidak masuk topik mana pun).

In [10]:
UMAP_PARAMS = dict(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED)
HDBSCAN_PARAMS = dict(min_cluster_size=15, min_samples=5, metric="euclidean", cluster_selection_method="eom", prediction_data=True)
VECTORIZER_PARAMS = dict(ngram_range=(1, 2), min_df=2)
CUSTOM_STOPWORDS = [
    "just", "like", "really", "video", "thanks", "thank", "dont", "don", "im", "ive", "youre",
    "lol", "yeah", "gonna", "got", "get", "make", "know", "think", "going", "people", "thing", "things",
]

In [11]:
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from bertopic import BERTopic

docs_df = df.loc[df["use_for_nlp"], ["comment_id", "text_clean"]].sort_values("comment_id").reset_index(drop=True)
docs = docs_df["text_clean"].tolist()

emb_model = SentenceTransformer(EMBEDDING_MODEL)
emb_path, emb_ids_path = CACHE / "embeddings.npy", CACHE / "embeddings_ids.csv"
cached = emb_path.exists() and emb_ids_path.exists() and not FORCE_RECOMPUTE_NLP
if cached and pd.read_csv(emb_ids_path, dtype=str)["comment_id"].tolist() == docs_df["comment_id"].tolist():
    embeddings = np.load(emb_path)
else:
    embeddings = emb_model.encode(docs, batch_size=64, show_progress_bar=True)
    np.save(emb_path, embeddings)
    docs_df[["comment_id"]].to_csv(emb_ids_path, index=False)

topic_model = BERTopic(
    embedding_model=emb_model,
    umap_model=UMAP(**UMAP_PARAMS),
    hdbscan_model=HDBSCAN(**HDBSCAN_PARAMS),
    vectorizer_model=CountVectorizer(stop_words=sorted(ENGLISH_STOP_WORDS.union(CUSTOM_STOPWORDS)), **VECTORIZER_PARAMS),
    top_n_words=10,
)
topics, _ = topic_model.fit_transform(docs, embeddings)
docs_df["topic_id"] = topics

topic_info = topic_model.get_topic_info()
topic_info["Representation"] = topic_info["Representation"].map(", ".join)
topic_info["Representative_Docs"] = topic_info["Representative_Docs"].map(lambda d: " || ".join(mask_mentions(x) for x in d) if isinstance(d, list) else "")
topic_info.to_csv(DATA_PROC / "topic_info.csv", index=False)
display(topic_info[["Topic", "Count", "Name", "Representation"]])

Batches: 100%|██████████| 183/183 [00:01<00:00, 154.24it/s]


,Topic,Count,Name,Representation
0,-1,5164,-1_ai_jobs_software_job,"ai, jobs, software, job, code, work, years, need, devin, good"
1,0,242,0_ai_ai ai_data_good ai,"ai, ai ai, data, good ai, problem, human, solve, intelligence, overlords, self"
2,1,182,1_vibe_vibe coding_coding_00,"vibe, vibe coding, coding, 00, 01, discussion, 02, code, vibe code, vibe coders"
3,2,176,2_devin_devin devin_hate devin_hate,"devin, devin devin, hate devin, hate, named, named devin, devin guy, devin hate, called devin, fuck devin"
4,3,168,3_fireship pill_fireship_pill_pills,"fireship pill, fireship, pill, pills, pilled, pill fireship, need fireship, doctor, want fireship, order"
...,...,...,...,...
141,140,16,140_fr_usual_didnt watch_melt,"fr, usual, didnt watch, melt, watching wish, inform, good sir, waiting, ll wait, infact"
142,141,15,141_13_success rate_threatened_pm,"13, success rate, threatened, pm, success, figure way, bare mind, better work, actual useful, 13 tasks"
143,142,15,142_therapist_pi_available_switching career,"therapist, pi, available, switching career, presume, 24 free, job create, happens ve, didn amazing, machines automate"
144,143,15,143_slop_ai slop_comment_porn,"slop, ai slop, comment, porn, complaining, ai generated, ship, architecture, generated, ai app"


### 5.1 Pemetaan topik ke klaster tema

Baca tabel `topic_info` di atas, lalu isi `THEME_MAP` dengan salah satu dari 4 klaster tema penelitian: **Tooling**, **Ekosistem**, **Risiko & Privasi**, **Dampak Industri**. Topik yang tidak cocok bisa diberi label **Lainnya**.

Pemetaan ini dilakukan manual, jadi catat alasannya di laporan (Bab III).

In [13]:
THEME_MAP = {
    # contoh: 0: "Tooling", 1: "Risiko & Privasi", 2: "Dampak Industri", 3: "Ekosistem",
}

docs_df["topic_label"] = docs_df["topic_id"].map(topic_info.set_index("Topic")["Name"])
docs_df["theme"] = [
    "Outlier" if t == -1 else THEME_MAP.get(t, "Belum dipetakan") for t in docs_df["topic_id"]
]

df = df.drop(columns=["topic_id", "topic_label", "theme"], errors="ignore").merge(
    docs_df[["comment_id", "topic_id", "topic_label", "theme"]], on="comment_id", how="left"
)
df["topic_id"] = df["topic_id"].astype("Int64")
print(df["theme"].value_counts(dropna=False))

theme
Belum dipetakan    6495
Outlier            5164
NaN                1648
Name: count, dtype: int64


## 6. Ekspor dataset

Semua file di `data/processed/` sudah dianonimkan dan aman di-upload ke GitHub.

| File | Isi | Dipakai untuk |
|---|---|---|
| `comments_clean.csv` | Satu baris per komentar, lengkap dengan sentimen dan topik | Sentimen, topik, RQ3 |
| `videos.csv` | Metadata video | Semua analisis |
| `edges_reply.csv` | Satu baris per balasan (Source membalas Target) | Analisis jaringan, RQ3 |
| `edges_reply_gephi.csv` | Edge berarah berbobot (Source, Target, Weight) | Gephi, Louvain, sentralitas |
| `nodes_users.csv` | Atribut tiap akun | Gephi (warna/ukuran node) |
| `edges_user_video.csv` | Edge pengguna–video berbobot | Graf bipartit, Jaccard |
| `nodes_bipartite.csv` | Node pengguna dan video beserta jenisnya | Gephi (graf bipartit) |
| `topic_info.csv` | Daftar topik BERTopic | Topik |
| `metadata.json` | Tanggal crawling, parameter, versi library | Reproducibility |

In [14]:
COMMENT_COLS = [
    "video_id", "comment_id", "parent_id", "is_reply", "user_id", "reply_to_user_id", "reply_target_method",
    "is_self_reply", "is_channel_owner", "text", "text_clean", "like_count", "total_reply_count",
    "published_at", "lang", "n_words", "use_for_nlp",
    "sentiment_label", "sentiment_score", "p_negative", "p_neutral", "p_positive",
    "topic_id", "topic_label", "theme",
]
comments_clean = df[COMMENT_COLS].sort_values(["video_id", "published_at", "comment_id"]).reset_index(drop=True)
comments_clean.to_csv(DATA_PROC / "comments_clean.csv", index=False)

videos = videos_raw.drop(columns=["crawled_at"])
videos.to_csv(DATA_PROC / "videos.csv", index=False)

# Edge balasan (per interaksi), tanpa balasan ke diri sendiri
rep = comments_clean[comments_clean["is_reply"] & comments_clean["reply_to_user_id"].notna() & ~comments_clean["is_self_reply"]]
edges_reply = rep.rename(columns={"user_id": "Source", "reply_to_user_id": "Target"})[
    ["Source", "Target", "video_id", "comment_id", "parent_id", "reply_target_method",
     "published_at", "sentiment_label", "sentiment_score", "topic_id", "theme"]
]
edges_reply.to_csv(DATA_PROC / "edges_reply.csv", index=False)

# Edge berbobot untuk Gephi (graf berarah)
edges_gephi = (
    edges_reply.groupby(["Source", "Target"])
    .agg(Weight=("comment_id", "size"), n_videos=("video_id", "nunique"), mean_sentiment=("sentiment_score", "mean"))
    .reset_index()
)
edges_gephi.insert(2, "Type", "Directed")
edges_gephi.to_csv(DATA_PROC / "edges_reply_gephi.csv", index=False)


def dominant(series):
    s = series.dropna()
    return s.value_counts().idxmax() if len(s) else pd.NA


# Atribut node pengguna
nodes = comments_clean.groupby("user_id").agg(
    n_comments=("comment_id", "size"),
    n_replies_sent=("is_reply", "sum"),
    n_videos=("video_id", "nunique"),
    total_likes=("like_count", "sum"),
    is_channel_owner=("is_channel_owner", "max"),
    mean_sentiment=("sentiment_score", "mean"),
    dominant_sentiment=("sentiment_label", dominant),
    dominant_theme=("theme", dominant),
).reset_index()
nodes["n_top_level"] = nodes["n_comments"] - nodes["n_replies_sent"]
nodes["n_replies_received"] = nodes["user_id"].map(edges_reply.groupby("Target").size()).fillna(0).astype(int)
nodes = nodes.rename(columns={"user_id": "Id"})
nodes.insert(1, "Label", nodes["Id"])
nodes.to_csv(DATA_PROC / "nodes_users.csv", index=False)

# Graf bipartit pengguna-video
edges_uv = comments_clean.groupby(["user_id", "video_id"]).size().reset_index(name="Weight")
edges_uv = edges_uv.rename(columns={"user_id": "Source", "video_id": "Target"})
edges_uv.insert(2, "Type", "Undirected")
edges_uv.to_csv(DATA_PROC / "edges_user_video.csv", index=False)

nodes_bip = pd.concat([
    pd.DataFrame({"Id": nodes["Id"], "Label": nodes["Id"], "node_type": "user"}),
    pd.DataFrame({"Id": videos["video_id"], "Label": videos["channel_title"], "node_type": "video"}),
], ignore_index=True)
nodes_bip.to_csv(DATA_PROC / "nodes_bipartite.csv", index=False)

for name, frame in {
    "comments_clean.csv": comments_clean, "videos.csv": videos, "edges_reply.csv": edges_reply,
    "edges_reply_gephi.csv": edges_gephi, "nodes_users.csv": nodes, "edges_user_video.csv": edges_uv,
    "nodes_bipartite.csv": nodes_bip, "topic_info.csv": topic_info,
}.items():
    print(f"{name:<24} {len(frame):>7} baris")

comments_clean.csv         13307 baris
videos.csv                     6 baris
edges_reply.csv             4869 baris
edges_reply_gephi.csv       4478 baris
nodes_users.csv            10371 baris
edges_user_video.csv       10531 baris
nodes_bipartite.csv        10377 baris
topic_info.csv               146 baris


In [15]:
from importlib.metadata import version

metadata = {
    "crawled_at": sorted(set(raw["crawled_at"].dropna())),
    "video_ids": videos["video_id"].tolist(),
    "n_rows_raw": int(len(raw)),
    "n_comments_clean": int(len(comments_clean)),
    "n_used_for_nlp": int(comments_clean["use_for_nlp"].sum()),
    "seed": SEED,
    "min_words_nlp": MIN_WORDS_NLP,
    "sentiment_model": SENTIMENT_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "umap_params": UMAP_PARAMS,
    "hdbscan_params": HDBSCAN_PARAMS,
    "vectorizer_params": {**VECTORIZER_PARAMS, "ngram_range": list(VECTORIZER_PARAMS["ngram_range"])},
    "custom_stopwords": CUSTOM_STOPWORDS,
    "theme_map": {str(k): v for k, v in THEME_MAP.items()},
    "python": platform.python_version(),
    "libraries": {p: version(p) for p in [
        "google-api-python-client", "pandas", "numpy", "langdetect", "torch", "transformers",
        "sentence-transformers", "bertopic", "umap-learn", "hdbscan", "scikit-learn",
    ]},
}
(DATA_PROC / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({k: metadata[k] for k in ["crawled_at", "n_rows_raw", "n_comments_clean", "n_used_for_nlp"]}, indent=2))

{
  "crawled_at": [
    "2026-11-19T04:31:14+00:00",
    "2026-11-19T04:31:20+00:00",
    "2026-11-19T04:31:27+00:00",
    "2026-11-19T04:31:31+00:00",
    "2026-11-19T04:31:34+00:00",
    "2026-11-19T04:31:37+00:00"
  ],
  "n_rows_raw": 13341,
  "n_comments_clean": 13307,
  "n_used_for_nlp": 11659
}


## 7. Pengecekan dataset

In [16]:
assert comments_clean["comment_id"].is_unique, "Ada comment_id ganda"
assert not comments_clean["user_id"].isna().any(), "Ada komentar tanpa user_id"
assert set(edges_gephi["Source"]) | set(edges_gephi["Target"]) <= set(nodes["Id"]), "Ada node di edge yang tidak ada di nodes_users"
assert (edges_gephi["Source"] != edges_gephi["Target"]).all(), "Masih ada self-loop"
assert set(edges_uv["Target"]) <= set(videos["video_id"]), "Ada video di edge bipartit yang tidak ada di videos"
assert not comments_clean["text"].str.contains(r"@(?!user\b)[^\s]+", regex=True).any(), "Masih ada @handle yang belum dianonimkan"

print("Semua pengecekan lolos.")
print(f"Komentar: {len(comments_clean)} | Pengguna: {len(nodes)} | Edge balasan: {len(edges_reply)} | Edge unik (Gephi): {len(edges_gephi)}")
display(pd.crosstab(comments_clean["video_id"], comments_clean["sentiment_label"].fillna("tidak dianalisis")))

Semua pengecekan lolos.
Komentar: 13307 | Pengguna: 10371 | Edge balasan: 4869 | Edge unik (Gephi): 4478


sentiment_label,negative,neutral,positive,tidak dianalisis
video_id,,,,
-5LfRL82Jck,204,263,151,71
AgyJv2Qelwk,3582,3174,1313,1175
JMYspR42HFM,173,169,107,69
PLKrSVuT-Dg,393,314,313,188
b0fv54xGOwY,164,174,98,35
tNmgmwEtoWE,458,331,278,110
